In [1]:
import scanpy as sc
import scipy.io as sio
import scipy.sparse as sp
import numpy as np
import pandas as pd

In [2]:
N4 = sc.read_h5ad("Bflo.N4_glia.h5ad")
T1 = sc.read_h5ad("Bflo.T1_glia.h5ad")

N4_spliced = sio.mmread("N4_glia_velo.spliced.mtx").tocsr()
N4_unspliced = sio.mmread("N4_glia_velo.unspliced.mtx").tocsr()
T1_spliced = sio.mmread("T1_glia_velo.spliced.mtx").tocsr()
T1_unspliced = sio.mmread("T1_glia_velo.unspliced.mtx").tocsr()

N4_genes = pd.read_csv("N4_glia_velo.genes.tsv", header=None)[0].astype(str).tolist()
N4_cells = pd.read_csv("N4_glia_velo.barcodes.tsv", header=None)[0].astype(str).tolist()

T1_genes = pd.read_csv("T1_glia_velo.genes.tsv", header=None)[0].astype(str).tolist()
T1_cells = pd.read_csv("T1_glia_velo.barcodes.tsv", header=None)[0].astype(str).tolist()

In [3]:
# function to add velocyte (Seurat format) information, spliced and unspliced matrix to h5ad objects 
def load_info(adata, gene, barcode, spliced, unspliced):
    common_genes = np.intersect1d(adata.var_names, gene)
    common_cells = np.intersect1d(adata.obs_names, barcode)
    gene_to_keep = pd.Index(gene).get_indexer(common_genes)
    cell_to_keep = pd.Index(barcode).get_indexer(common_cells)

    spliced = spliced[:, cell_to_keep]
    spliced = spliced[gene_to_keep,:]
    
    unspliced = unspliced[:, cell_to_keep]
    unspliced = unspliced[gene_to_keep,:]
    
    adata = adata[common_cells, common_genes].copy()
    
    adata.layers["spliced"] = spliced.T
    adata.layers["unspliced"] = unspliced.T
    return adata

In [4]:
N4 = load_info(N4, N4_genes, N4_cells, N4_spliced, N4_unspliced)
T1 = load_info(T1, T1_genes, T1_cells, T1_spliced, T1_unspliced)

In [5]:
N4.write_h5ad("Bflo.N4_glia.velo_added.h5ad")
T1.write_h5ad("Bflo.T1_glia.velo_added.h5ad")